# EasyMagpieTTS — vLLM-Omni inference demo

End-to-end inference through `easymagpie_vllm_omni` on a checkpoint converted by
`easy_magpietts_convert_to_vllm.py`: prefill (speaker embedding + context text) ->
autoregressive decode (target text, one subword per step) -> stacked audio codes ->
vocode to waveform.

Run inside the `vllm_omni_env` with the plugin installed
(`pip install -e examples/tts/easymagpie_vllm_omni`).

In [ ]:
import os

os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import json
import tempfile
import uuid
from pathlib import Path

import torch
import yaml

from vllm import SamplingParams
from vllm.sampling_params import RequestOutputKind
from vllm_omni import AsyncOmni

from easymagpie_vllm_omni.config import EasyMagpieOmniArch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Converted model directory

`MODEL_DIR` is the directory written by the converter: `config.json`, weights, the
text tokenizer, and per-speaker embeddings. The engine loads everything from it;
here we only read a few config scalars used to build the prompt.

In [ ]:
MODEL_DIR = Path("/home/vklimkov/workspace/emp/NeMo/examples/tts/easymagpie_vllm_omni/easymp_vllm_model")
assert (MODEL_DIR / "config.json").exists(), f"No config.json under {MODEL_DIR}; run the converter first."

config = json.loads((MODEL_DIR / "config.json").read_text())
arch = EasyMagpieOmniArch.from_hf_config(type("Cfg", (), config))

TEXT_VOCAB = int(config["text_vocab_size"])
TEXT_EOS_ID = TEXT_VOCAB - 2
AUDIO_STOP_TOKEN_ID = max(1, int(config.get("vocab_size", 2)) - 1)
SPEECH_DELAY = int(getattr(arch, "streaming_speech_delay", 0) or 0)

print(f"Model dir                : {MODEL_DIR}")
print(f"num_stacked_codebooks    : {arch.num_stacked_codebooks}  (C*S)")
print(f"audio_bos / audio_eos id : {arch.audio_bos_id} / {arch.audio_eos_id}")
print(f"text_vocab / text_eos    : {TEXT_VOCAB} / {TEXT_EOS_ID}")
print(f"audio-EOS stop token id  : {AUDIO_STOP_TOKEN_ID}")
print(f"streaming speech delay   : {SPEECH_DELAY} frames")

## 2. Single-stage `AsyncOmni` engine

One `llm` stage running the EasyMagpie talker (`worker_type="ar"`).
`final_output_type="audio"` makes the runner attach the per-step `audio_codes`
payload to each output. `async_chunk=True` lets the streaming-text path (§5) feed
one subword per step and is a no-op for the whole-text path (§4), so a single
engine serves both.

In [ ]:
DECODE_STEPS = 256       # max audio frames to decode (trimmed at audio EOS)
MAX_MODEL_LEN = 1024
MAX_NUM_BATCHED_TOKENS = 1024

stage_cfg = {
    "async_chunk": True,
    "stage_args": [
        {
            "stage_id": 0,
            "stage_type": "llm",
            "is_comprehension": True,
            "final_output": True,
            "final_output_type": "audio",
            "runtime": {"devices": "0"},
            "engine_args": {
                "model_stage": "easymagpie",
                "max_num_seqs": 1,
                "model_arch": "EasyMagpieTTSForConditionalGeneration",
                "worker_type": "ar",
                "scheduler_cls": "easymagpie_vllm_omni.scheduler.EasyMagpieARAsyncScheduler",
                "trust_remote_code": True,
                "async_scheduling": True,
                "enable_prefix_caching": False,
                "enforce_eager": False,
                "engine_output_type": "audio",
                "gpu_memory_utilization": 0.8,
                "distributed_executor_backend": "uni",
                "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
                "max_model_len": MAX_MODEL_LEN,
                "dtype": "float16",
                "mamba_ssm_cache_dtype": "float32",
                "attention_backend": "TRITON_ATTN",
                "skip_tokenizer_init": True,
            },
            "default_sampling_params": {
                "temperature": 0.0,
                "max_tokens": DECODE_STEPS,
                "detokenize": False,
                "ignore_eos": True,
                "stop_token_ids": [AUDIO_STOP_TOKEN_ID],
            },
        }
    ],
}

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", prefix="easymagpie_omni_demo_", delete=False,
)
yaml.dump(stage_cfg, _tmp, sort_keys=False)
_tmp.close()
STAGE_CFG_PATH = _tmp.name
print(f"Stage config: {STAGE_CFG_PATH}")

omni = AsyncOmni(
    model=str(MODEL_DIR),
    stage_configs_path=STAGE_CFG_PATH,
    log_stats=False,
    stage_init_timeout=300,
)
print("Engine ready (single stage: EasyMagpie talker)")

## 3. Build the prompt

Per-request inputs passed via `additional_information`: `speaker_embedding`
`(T_audio, embedding_dim)`, the `context_text` string, and the target `text` (all
tokenized in-engine). `prompt_token_ids` are placeholders sized by
`estimate_prompt_len(...)` to match the assembled prefill context. Audio sampling
`temperature` / `top_k` are forwarded to the local transformer.

In [ ]:
torch.manual_seed(0)

from transformers import AutoTokenizer

from easymagpie_vllm_omni.easymagpie import EasyMagpieTTSForConditionalGeneration

SPEAKER_NAME = "eng"
_loaded = torch.load(MODEL_DIR / "speaker_embeddings" / f"{SPEAKER_NAME}.pt", map_location="cpu")
speaker_embedding = _loaded["speaker_encoding"] if isinstance(_loaded, dict) else _loaded
speaker_embedding = speaker_embedding.to(torch.float32)

CONTEXT_TEXT = "[EN]"
TEXT = "Hello, this is a test of the EasyMagpie text to speech model."

# Audio sampling (local transformer); temperature=0 == argmax (deterministic).
LT_TEMPERATURE = 0.0
LT_TOPK = 80

# Same tokenizer the engine loads from MODEL_DIR; used only to size the placeholders.
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
prompt_len = EasyMagpieTTSForConditionalGeneration.estimate_prompt_len(
    speaker_embedding,
    tokenize=lambda t: tokenizer.encode(t),
    context_text=CONTEXT_TEXT,
    has_task_embedding=arch.num_task_embeddings > 0,
)

prompt = {
    "prompt_token_ids": [0] * prompt_len,
    "additional_information": {
        "speaker_embedding": speaker_embedding,
        "context_text": CONTEXT_TEXT,
        "text": TEXT,
        "temperature": LT_TEMPERATURE,
        "top_k": LT_TOPK,
    },
}
assert prompt_len + DECODE_STEPS <= MAX_MODEL_LEN

# output_kind=DELTA: audio_codes arrives as a growing list during decode
# ([prefill_prefix, frame_0, frame_1, ...]); prefill/final steps yield a tensor.
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=DECODE_STEPS,
    detokenize=False,
    ignore_eos=True,
    stop_token_ids=[AUDIO_STOP_TOKEN_ID],
    output_kind=RequestOutputKind.DELTA,
)

print(f"speaker_embedding         : {tuple(speaker_embedding.shape)}")
print(f"context_text / text       : {CONTEXT_TEXT!r} / {TEXT!r}")
print(f"prompt_len (placeholders) : {prompt_len}")

## 4. Whole-text inference -> audio codes

`omni.generate(...)` yields one `RequestOutput` per engine step, each carrying the
**cumulative** `multimodal_output["audio_codes"]` tensor: the first `prompt_len`
rows are the prefill prefix, followed by one `(1, C*S)` decoded frame per step
(occasionally surfaced as a list of per-step tensors, which we just concatenate).
We keep the largest tensor seen, drop the prefill prefix, then drop the
`speech_delay` warm-up frames and the trailing audio-EOS frame.

In [ ]:
async def run_request(prompt, sampling_params):
    """Keep the cumulative audio-code tensor [prefill_prefix | decoded frames].

    Each step exposes audio_codes as a growing tensor (sometimes a list of per-step
    tensors, which we concatenate); we keep the largest one seen.
    """
    request_id = f"easymagpie-{uuid.uuid4().hex[:8]}"
    codes = None
    async for out in omni.generate(prompt, sampling_params_list=[sampling_params], request_id=request_id):
        payload = (out.multimodal_output or {}).get("audio_codes")
        if isinstance(payload, list):
            payload = torch.cat([t for t in payload if isinstance(t, torch.Tensor)], dim=0) if payload else None
        if isinstance(payload, torch.Tensor) and (codes is None or payload.shape[0] >= codes.shape[0]):
            codes = payload.detach().cpu().to(torch.long)
    return codes


codes = await run_request(prompt, sampling_params)
# Rows 0:prompt_len are the prefill prefix; decoded frames follow. Drop the prefix,
# the speech-delay warm-up frames, and the trailing audio-EOS frame.
audio_codes = codes[prompt_len + SPEECH_DELAY : -1]
print(f"cumulative codes : {tuple(codes.shape)}  (prompt_len={prompt_len})")
print(f"audio_codes      : {tuple(audio_codes.shape)}  (dropped prefix + {SPEECH_DELAY} warm-up + 1 EOS)")

In [ ]:
import matplotlib.pylab as plt

plt.figure(figsize=(10, 3))
plt.imshow(audio_codes.T, aspect="auto", interpolation="nearest")
plt.title(f"whole-text audio codes  ({audio_codes.shape[0]} frames)")
plt.xlabel("decode frame")
plt.ylabel("stacked codebook")
plt.colorbar()
plt.show()

## 5. Streaming text input (one subword per decode step)

Same model, but push subword ids **one at a time** as it decodes (the live-client
path). `omni.generate(...)` is given an async generator of `StreamingInput` chunks:
chunk 0 is the prefill (speaker + context, no text), each later chunk carries one
`text_token` with `max_tokens=1` (one chunk -> one decoded frame). After the text
is exhausted we feed the `-1` mask sentinel until the model emits the audio-EOS
token. Input is paced by output via `go_queue`. Codes are collected exactly as in
§4 (keep the cumulative tensor, slice off prefix + warm-up + EOS).

In [ ]:
import asyncio
from collections.abc import AsyncGenerator

try:
    from vllm.engine.protocol import StreamingInput
except ImportError:
    from vllm.v1.engine.async_llm import StreamingInput

# Tokenize the target text the same way the model does (specials off + text-EOS).
text_ids = list(tokenizer.encode(TEXT, add_special_tokens=False)) + [TEXT_EOS_ID]
print(f"streamed text ids ({len(text_ids)}): {text_ids}")

stream_params = SamplingParams(
    temperature=0.0,
    max_tokens=1,
    detokenize=False,
    ignore_eos=True,
    stop_token_ids=[AUDIO_STOP_TOKEN_ID],
    output_kind=RequestOutputKind.DELTA,
)

go_queue: asyncio.Queue[bool] = asyncio.Queue()


async def stream_text_inputs() -> AsyncGenerator[StreamingInput, None]:
    # Prefill: speaker + context, NO text (its absence selects streaming-text mode).
    # The first subword rides on the first decode chunk, not the prefill.
    prefill_info = {
        "speaker_embedding": speaker_embedding,
        "context_text": CONTEXT_TEXT,
        "temperature": LT_TEMPERATURE,
        "top_k": LT_TOPK,
    }
    yield StreamingInput(
        prompt={"prompt_token_ids": [0] * prompt_len, "additional_information": prefill_info},
        sampling_params=stream_params,
    )
    # One chunk per decode frame, paced by go_queue: text_ids[k] on decode step k,
    # then the -1 mask once the text is exhausted.
    step = 0
    while True:
        await go_queue.get()
        tok = int(text_ids[step]) if step < len(text_ids) else -1
        yield StreamingInput(
            prompt={"prompt_token_ids": [0], "additional_information": {"text_token": tok}},
            sampling_params=stream_params,
        )
        step += 1


async def run_streaming_request():
    """Collect codes like §4, pacing one chunk per decoded frame.

    Every streaming segment reports finish_reason "length"; the request truly ends
    only at the audio-EOS stop token, where we stop pacing and break.
    """
    request_id = f"easymagpie-stream-{uuid.uuid4().hex[:8]}"
    codes = None
    MAX_STEPS = 4 * DECODE_STEPS + 16
    steps = 0
    async for out in omni.generate(
        stream_text_inputs(), sampling_params_list=[stream_params], request_id=request_id
    ):
        steps += 1
        co = out.outputs[0] if out.outputs else None
        payload = (out.multimodal_output or {}).get("audio_codes")
        if isinstance(payload, list):
            payload = torch.cat([t for t in payload if isinstance(t, torch.Tensor)], dim=0) if payload else None
        if isinstance(payload, torch.Tensor) and (codes is None or payload.shape[0] >= codes.shape[0]):
            codes = payload.detach().cpu().to(torch.long)
        if getattr(co, "stop_reason", None) == AUDIO_STOP_TOKEN_ID or steps >= MAX_STEPS:
            break
        await go_queue.put(True)
    return codes


codes_stream = await run_streaming_request()
audio_codes_stream = codes_stream[prompt_len + SPEECH_DELAY : -1]
print(f"cumulative codes : {tuple(codes_stream.shape)}")
print(f"audio_codes      : {tuple(audio_codes_stream.shape)}")

## 6. Decode codes to waveforms

The engine emits **stacked** codes `(T, C*S)` with `C*S = 16`. To vocode we mirror
`EasyMagpieTTSInferenceModel`: load the `.nemo` codec, unstack `(T, C*S)` ->
`(1, C, T*S)`, optionally remap the regrouped FSQ token space back to the codec's
native space, then `codec_model.decode(...)`.

Set `CODEC_MODEL_PATH` / `EASYMAGPIE_NEMO` to the same `.nemo` files passed to the
converter. Needs NeMo importable in this environment.

In [ ]:
from hydra.utils import instantiate
from IPython.display import Audio, display

from nemo.collections.tts.models import AudioCodecModel
from nemo.collections.tts.models.easy_magpietts_inference import EasyMagpieTTSInferenceModel
from nemo.collections.tts.modules.audio_codec_modules import VectorQuantizerIndexConverter

CODEC_MODEL_PATH = "/home/vklimkov/workspace/emp/ckpt/easymagpietts_NEXT/25fps_spectral_codec_with_bandwidth_extension.nemo"
EASYMAGPIE_NEMO = "/home/vklimkov/workspace/emp/ckpt/easymagpietts_NEXT/2605_NemotronTTS_V0.2/v2/2605_EMTTS_SmallMamba_Step150k_posttrained_epoch12.nemo"

# Load the codec once (drop the discriminator to save memory).
_codec_cfg = AudioCodecModel.restore_from(CODEC_MODEL_PATH, return_config=True)
if "use_scl_loss" in _codec_cfg:
    _codec_cfg.use_scl_loss = False
codec_model = AudioCodecModel.restore_from(CODEC_MODEL_PATH, strict=False, override_config_path=_codec_cfg)
if hasattr(codec_model, "discriminator"):
    del codec_model.discriminator
codec_model = codec_model.eval().to("cuda" if torch.cuda.is_available() else "cpu")
codec_device = next(codec_model.parameters()).device

# The model may use a regrouped FSQ token space; map it back to the codec's native
# space when they differ (config read from the source EasyMagpie .nemo, no weights).
_em_cfg = EasyMagpieTTSInferenceModel.restore_from(EASYMAGPIE_NEMO, return_config=True)
_vq_cfg = _em_cfg.get("vector_quantizer")
if _vq_cfg is not None and instantiate(_vq_cfg).num_codebooks != codec_model.vector_quantizer.num_codebooks:
    codec_converter = VectorQuantizerIndexConverter(
        vector_quantizer_original=codec_model.vector_quantizer,
        vector_quantizer_new=instantiate(_vq_cfg),
    ).to(codec_device)
else:
    codec_converter = None
print(f"codec native codebooks : {codec_model.vector_quantizer.num_codebooks}")
print(f"codec token converter  : {'enabled' if codec_converter is not None else 'not needed'}")

S = arch.frame_stacking_factor
C = arch.num_stacked_codebooks // S
sample_rate = int(codec_model.output_sample_rate)


def decode_codes_to_waveform(audio_codes: torch.Tensor):
    """Decode one (T, C*S) stacked-code sequence to a mono float32 waveform."""
    stacked = audio_codes.to(codec_device, torch.long).T.unsqueeze(0)  # (1, C*S, T)
    T_out = stacked.size(-1)
    codes = stacked.view(1, C, S, T_out).permute(0, 1, 3, 2).reshape(1, C, T_out * S)  # (1, C, T*S)
    codes_len = torch.tensor([codes.size(-1)], device=codec_device, dtype=torch.long)

    MIN_LEN = 4
    if int(codes_len.min()) < MIN_LEN:
        codes = torch.nn.functional.pad(codes, (0, MIN_LEN - int(codes_len.min())), value=0)
        codes_len = codes_len.clamp(min=MIN_LEN)
    codes = codes.clamp_(0, arch.codebook_size - 1)

    with torch.no_grad(), torch.autocast(device_type=codec_device.type, dtype=torch.float32):
        if codec_converter is not None:
            codes = codec_converter.convert_new_to_original(audio_tokens=codes, audio_lens=codes_len)
        audio, audio_len = codec_model.decode(tokens=codes, tokens_len=codes_len)
    return audio[0, : int(audio_len[0])].detach().cpu().float().numpy()

In [ ]:
# Vocode and play each run.
for name, codes in [("whole-text (§4)", audio_codes), ("streamed text (§5)", audio_codes_stream)]:
    wav = decode_codes_to_waveform(codes)
    print(f"{name:<20}: {tuple(codes.shape)} codes -> {wav.shape[0] / sample_rate:.2f}s @ {sample_rate} Hz")
    display(Audio(wav, rate=sample_rate))

### Streamed-text audio codes

In [ ]:
import matplotlib.pylab as plt

plt.figure(figsize=(10, 3))
plt.imshow(audio_codes_stream.T, aspect="auto", interpolation="nearest")
plt.title(f"streamed-text audio codes  ({audio_codes_stream.shape[0]} frames)")
plt.xlabel("decode frame")
plt.ylabel("stacked codebook")
plt.colorbar()
plt.show()